# 04 — Windows Dataset Explorer (P2)

Everything in this notebook is derived directly from `windows.csv` / `windows.parquet` —
no saved triage file, no number carried over from any prior analysis session is trusted
as an input. Where a section's conclusion happens to match an earlier informal finding,
that is reported as a cross-check, not assumed in advance.

**Scope.** `02`/`03` work at the raw-event level to separate genuine behavioural signal
from quantisation artifact. This notebook works one layer up, at the window level, and
asks a different question: *is this dataset, as built, actually ready to model, and which
specific features can be trusted to carry person-signal rather than device or transient-
state signal?*

**House rule for this notebook:** every section states what it computes, computes it live
against the loaded file, and prints the result before drawing any conclusion. Sections
that cannot be answered with the current cohort say so explicitly (`UNTESTABLE`) rather
than falling back to a weaker, confounded comparison.

## Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

IN_CSV = Path("data/processed/windows.csv")
IN_PARQUET = Path("data/processed/windows.parquet")

def load_windows():
    if IN_PARQUET.exists():
        return pd.read_parquet(IN_PARQUET)
    if IN_CSV.exists():
        return pd.read_csv(IN_CSV, low_memory=False)
    raise FileNotFoundError(f"Could not find {IN_PARQUET} or {IN_CSV}")

df = load_windows()
print(f"{len(df):,} windows, {len(df.columns)} columns")

FileNotFoundError: Could not find data/processed/windows.parquet or data/processed/windows.csv

## 0 — Load and orient

**Goal:** establish what is actually in the file before analysing anything, including
catching unit/scale mistakes early (e.g. a `_pct` column that is actually a 0–1 fraction)
rather than discovering them three sections later.

In [ ]:
print("dtype breakdown:")
print(df.dtypes.value_counts().to_string())

dtype breakdown:
float64    545
int64       60
str         12
bool         9


In [ ]:
cohort = (df.groupby("participantId")
            .agg(sessions=("sessionId", "nunique"),
                 windows=("sessionId", "size"),
                 deviceModels=("deviceModel", lambda s: sorted(s.unique())),
                 deviceFamily=("deviceFamily", "first"))
            .sort_values("sessions", ascending=False))
print("=== COHORT TABLE (computed live, reused by every later section) ===")
print(cohort.to_string())

=== COHORT TABLE (computed live, reused by every later section) ===
               sessions  windows                 deviceModels deviceFamily
participantId                                                             
p3YDMHG               2      281             [iphone_16_plus]       mobile
pA6XL23               2      249                  [iphone_16]       mobile
pG5G4MS               2      105             [iphone_16_plus]       mobile
pCURDWC               1       72           [macbook_trackpad]      desktop
pG2C8BC               1       84  [windows_laptop_with_mouse]      desktop
pZMC82M               1      172                  [iphone_15]       mobile


In [ ]:
# Full describe() over every numeric column, unfiltered. This is the check that
# would have caught a column named "_pct" actually ranging 0-1 on day one: the min/max
# columns below make a unit mismatch visible immediately, before any downstream section
# builds an assumption on top of it.
numeric_cols = df.select_dtypes(include=[np.number]).columns
desc = df[numeric_cols].describe().T[["min", "25%", "50%", "75%", "max"]]
suspect = desc[(desc["min"] >= 0) & (desc["max"] <= 1) & (desc["max"] > 0)]
print(f"{len(suspect)} numeric columns range entirely within [0, 1] — confirm each of these "
      f"is genuinely a fraction/probability and not a mislabelled percentage:")
print(suspect.index.tolist())

90 numeric columns range entirely within [0, 1] — confirm each of these is genuinely a fraction/probability and not a mislabelled percentage:
['taskPurity', 'typing_press_to_press_ms_slope_r2', 'typing_dwell_ms_slope_r2', 'typing_delta_length_slope_r2', 'touch_speed_slope_r2', 'touch_bearing_consistency', 'touch_radiusX_cv', 'touch_radiusX_slope_r2', 'touch_force_cv', 'touch_touchesCount_mean', 'touch_touchesCount_median', 'touch_touchesCount_p95', 'touch_touchesCount_max', 'touch_touchesCount_min', 'pointer_speed_slope_r2', 'pointer_bearing_consistency', 'pointer_pressure_mean', 'pointer_pressure_std', 'pointer_pressure_p95', 'pointer_pressure_max', 'pointer_pressure_burstiness', 'pointer_pressure_local_inconsistency', 'pointer_pressure_slope_r2', 'pointer_width_cv', 'pointer_height_cv', 'scroll_velocity_slope_r2', 'motion_coverage_pct', 'motion_ax_std', 'motion_ax_iqr', 'motion_ax_local_inconsistency', 'motion_ax_slope_r2', 'motion_ay_std', 'motion_ay_iqr', 'motion_ay_local_inconsist

**Read this as:** the cohort table above is the one place session/participant/device
counts are computed in this notebook — every later section (§3, §5, §8) refers back to it
rather than re-deriving it, so a stale number can't quietly diverge from a fresh one.

## 1 — Column census

**Goal:** partition every column into **metadata**, **reliability**, or **behavioural**,
using only name pattern and dtype — not a memory of what earlier analysis called each
column. The rule for each bucket is printed so it is auditable against the actual header.

- **Metadata**: identity/session/device/schema/consent/task-context fields, plus any
  `ctx*` context-questionnaire columns present (there may be zero — this file may predate
  the questionnaire, or a future run may add more fields than are listed here).
- **Reliability**: numeric columns ending in `_n`, or containing `coverage` — these
  describe how much data a window's estimate rests on, not the estimate itself. They
  belong in a fusion-confidence layer, never in a feature vector fed to a scorer.
- **Presence flags**: boolean `has*` columns — modality-active indicators, same role as
  reliability but boolean rather than continuous.
- **Behavioural**: everything numeric left over. This is the only bucket eligible for
  §2 onward.

In [ ]:
META_COLS_KNOWN = [
    "sessionId", "participantId", "sessionIndex", "windowIndex", "windowStartMs", "windowEndMs",
    "identitySource", "schemaVersion", "appVersion", "deviceFamily", "devicePlatform", "deviceModel",
    "consentVersion", "completedNormally", "usableForSignalExtraction", "activityType",
    "dominantTaskId", "dominantBaseTaskId", "taskPurity", "n_events_in_window", "n_distinct_tasks_in_window",
]
CTX_COLS = [c for c in df.columns if c.startswith("ctx")]
PRESENCE_FLAGS = [c for c in df.columns if c.startswith("has") and df[c].dtype == bool]

META_ALL = [c for c in META_COLS_KNOWN if c in df.columns] + CTX_COLS

REL_COLS = [c for c in df.columns
            if c not in META_ALL and c not in PRESENCE_FLAGS
            and (c.endswith("_n") or "coverage" in c.lower())]

BEHAV_COLS = [c for c in df.select_dtypes(include=[np.number]).columns
              if c not in META_ALL and c not in REL_COLS and c not in PRESENCE_FLAGS]

unaccounted = [c for c in df.columns
               if c not in META_ALL + PRESENCE_FLAGS + REL_COLS + BEHAV_COLS]

print(f"metadata        : {len(META_ALL):>4}  (of which {len(CTX_COLS)} are ctx* context fields)")
print(f"presence flags  : {len(PRESENCE_FLAGS):>4}")
print(f"reliability     : {len(REL_COLS):>4}")
print(f"behavioural     : {len(BEHAV_COLS):>4}")
print(f"unaccounted     : {len(unaccounted):>4}  {unaccounted}")
print(f"total           : {len(META_ALL)+len(PRESENCE_FLAGS)+len(REL_COLS)+len(BEHAV_COLS)+len(unaccounted)} "
      f"of {len(df.columns)} columns")

if len(CTX_COLS) == 0:
    print("\nNote: 0 ctx* columns found. Either this file predates the context questionnaire, "
          "or context fields use different naming — check manually if unexpected.")

metadata        :   21  (of which 0 are ctx* context fields)
presence flags  :    7
reliability     :   44
behavioural     :  554
unaccounted     :    0  []
total           : 626 of 626 columns

Note: 0 ctx* columns found. Either this file predates the context questionnaire, or context fields use different naming — check manually if unexpected.


**Read this as:** `unaccounted` should be empty. If it isn't, a new column exists in
`windows.csv` that this census doesn't yet have a rule for — extend the rule above rather
than letting it silently fall into `BEHAV_COLS` by accident.

## 2 — Dead and sparse features

**Goal:** find zero-information behavioural columns, computed fresh — not loaded from any
saved triage file. A column is `DEAD` if it never varies; `SPARSE` if it's populated in
under 5% of windows.

In [ ]:
def census_row(s):
    nn = s.notna().sum()
    return dict(pct_populated=round(100 * nn / len(s), 2), n_unique=int(s.nunique(dropna=True)))

census = pd.DataFrame({c: census_row(df[c]) for c in BEHAV_COLS}).T
census.index.name = "col"

dead = census[census.n_unique <= 1]
sparse = census[(census.pct_populated < 5) & (census.n_unique > 1)]
live = census[~census.index.isin(dead.index) & ~census.index.isin(sparse.index)]

print(f"DEAD (zero variance)      : {len(dead)}")
print(f"SPARSE (<5% populated)    : {len(sparse)}")
print(f"LIVE (survives this gate) : {len(live)}  of {len(BEHAV_COLS)} behavioural columns")

LIVE_COLS = live.index.tolist()

DEAD (zero variance)      : 84
SPARSE (<5% populated)    : 26
LIVE (survives this gate) : 444  of 554 behavioural columns


**Read this as:** `LIVE_COLS` is the working feature set for every section from here
on. `dead`/`sparse` feed `build_behavioural_dataset.py`'s drop list directly — no further
judgement needed on these, they carry no information regardless of what §4/§5 find.

## 3 — Device-family and device-model coverage

**Goal:** map which features exist per device family, and — the constraint that governs
every later section — which device models actually have more than one participant. A
feature can only be tested against a device-vs-person confound where a shared-device
group exists; everywhere else, that question is structurally unanswerable with this
cohort, not just weakly answerable.

In [ ]:
by_family = df.groupby("deviceFamily")[LIVE_COLS].apply(lambda g: g.notna().any())
fam_coverage = by_family.T
fam_coverage.columns = [f"available_on_{c}" for c in fam_coverage.columns]
mobile_only = fam_coverage[fam_coverage.get("available_on_mobile", False)
                            & ~fam_coverage.get("available_on_desktop", True)].index.tolist()
desktop_only = fam_coverage[fam_coverage.get("available_on_desktop", False)
                             & ~fam_coverage.get("available_on_mobile", True)].index.tolist()
shared = [c for c in LIVE_COLS if c not in mobile_only and c not in desktop_only]

print(f"mobile-only features  : {len(mobile_only)}")
print(f"desktop-only features : {len(desktop_only)}")
print(f"shared across families: {len(shared)}")

mobile-only features  : 258
desktop-only features : 0
shared across families: 186


In [ ]:
by_model = df.groupby("deviceModel")["participantId"].nunique()

print("=== distinct participants per device model (computed live) ===")
print(by_model.sort_values(ascending=False).to_string())

TESTABLE_MODELS = by_model[by_model >= 2].index.tolist()
UNTESTABLE_MODELS = by_model[by_model < 2].index.tolist()
print(f"\nTestable for device-vs-person (>=2 participants): {TESTABLE_MODELS}")
print(f"Untestable (1 participant only)                 : {UNTESTABLE_MODELS}")
print(f"\n-> The device-vs-person question in \u00a75 can only be answered for "
      f"{len(TESTABLE_MODELS)} of {len(by_model)} device models this cohort has used.")

=== distinct participants per device model (computed live) ===
deviceModel
iphone_16_plus               2
iphone_15                    1
iphone_16                    1
macbook_trackpad             1
windows_laptop_with_mouse    1

Testable for device-vs-person (>=2 participants): ['iphone_16_plus']
Untestable (1 participant only)                 : ['iphone_15', 'iphone_16', 'macbook_trackpad', 'windows_laptop_with_mouse']

-> The device-vs-person question in §5 can only be answered for 1 of 5 device models this cohort has used.


**Read this as:** the `TESTABLE_MODELS` list is not a detail — it is the hard ceiling
on how much of §5's output can be trusted. Recruiting a second participant onto any
currently single-occupant device model is worth more to this specific question than
another session from an existing participant.

## 4 — Quantisation and base-unit recovery

**Goal:** find features whose apparent structure is a hardware constant, then test
whether normalising by a device-specific base unit recovers real person-signal
underneath — the general form of a finding already confirmed once for `pointer_width`
(quantised per device model, but the *distribution over quanta* separated two people
sharing the same model). This function is written to run on any quantised feature it
finds, not applied only to the one case already known.

In [ ]:
def quantisation_report(s, n_top_values=6):
    vals = s.dropna()
    if vals.empty:
        return None
    vc = vals.value_counts()
    return dict(n_unique=int(vals.nunique()), top_values=vc.head(n_top_values).index.tolist())

quant_candidates = live[live.n_unique.between(2, 20)].index.tolist()
print(f"{len(quant_candidates)} LIVE features with 2-20 distinct values — quantisation candidates")

51 LIVE features with 2-20 distinct values — quantisation candidates


In [ ]:
def base_unit_recovery_test(col, testable_models=TESTABLE_MODELS):
    """For a quantised column: does its base unit vary by deviceModel (device leak)?
    Within a testable (>=2 participant) model, does the distribution *over* quanta still
    separate participants (recoverable signal)? Returns None if no testable model exists
    for this column."""
    per_model_units = df.groupby("deviceModel")[col].apply(
        lambda s: sorted(s.dropna().unique())[:3])
    if not testable_models:
        return dict(col=col, status="UNTESTABLE_NO_SHARED_DEVICE", per_model_units=per_model_units.to_dict())

    results = []
    for model in testable_models:
        sub = df[df.deviceModel == model]
        pids = sub.participantId.unique()
        if len(pids) != 2:
            continue
        a = sub.loc[sub.participantId == pids[0], col].dropna().values.astype(float)
        b = sub.loc[sub.participantId == pids[1], col].dropna().values.astype(float)
        if len(a) < 10 or len(b) < 10:
            continue
        combined = np.r_[a, b]
        if np.std(combined) == 0:
            results.append(dict(model=model, pair=list(pids), within_model_auc=np.nan))
            continue
        r = rankdata(combined); n1 = len(a)
        u = r[:n1].sum() - n1 * (n1 + 1) / 2
        auc = max(u / (n1 * len(b)), 1 - u / (n1 * len(b)))
        results.append(dict(model=model, pair=list(pids), within_model_auc=round(auc, 3)))

    return dict(col=col, status="TESTED", per_model_units=per_model_units.to_dict(), results=results)

# Run on every quantisation candidate that is also flagged mobile-only/shared (device-relevant)
quant_reports = [base_unit_recovery_test(c) for c in quant_candidates]
recoverable = [r for r in quant_reports if r.get("status") == "TESTED"
               and any((x.get("within_model_auc") or 0) >= 0.80 for x in r["results"])]
print(f"Quantised features with a testable shared-device group: "
      f"{sum(1 for r in quant_reports if r['status']=='TESTED')}")
print(f"Of those, features with within-model separation >= 0.80 (recoverable signal): {len(recoverable)}")
for r in recoverable[:10]:
    print(f"  {r['col']}: {r['results']}")

Quantised features with a testable shared-device group: 51
Of those, features with within-model separation >= 0.80 (recoverable signal): 11
  touch_n_touchstart: [{'model': 'iphone_16_plus', 'pair': ['p3YDMHG', 'pG5G4MS'], 'within_model_auc': np.float64(0.872)}]
  touch_n_touchend: [{'model': 'iphone_16_plus', 'pair': ['p3YDMHG', 'pG5G4MS'], 'within_model_auc': np.float64(0.868)}]
  pointer_n_pointerdown: [{'model': 'iphone_16_plus', 'pair': ['p3YDMHG', 'pG5G4MS'], 'within_model_auc': np.float64(0.872)}]
  pointer_width_median: [{'model': 'iphone_16_plus', 'pair': ['p3YDMHG', 'pG5G4MS'], 'within_model_auc': np.float64(0.819)}]
  pointer_width_max: [{'model': 'iphone_16_plus', 'pair': ['p3YDMHG', 'pG5G4MS'], 'within_model_auc': np.float64(0.871)}]
  pointer_height_median: [{'model': 'iphone_16_plus', 'pair': ['p3YDMHG', 'pG5G4MS'], 'within_model_auc': np.float64(0.819)}]
  pointer_height_max: [{'model': 'iphone_16_plus', 'pair': ['p3YDMHG', 'pG5G4MS'], 'within_model_auc': np.float64(0.8

**Read this as:** a feature landing in `recoverable` failed the naive dead-feature
test for the right reason — real signal riding on a device-specific scale — and should be
re-expressed as an integer quantum count (value ÷ its device model's base unit) rather
than dropped. Everything else in `quant_candidates` that is *not* in `recoverable` is
either a genuine device constant, or untestable for lack of a shared-device group — the
next section states which.

## 5 — Device-vs-person decomposition

**Goal:** classify every LIVE feature into one of six categories, so that "this feature
separates people" is never reported without also reporting whether that separation
survives a device control and a within-person stability control. This directly answers
the question this notebook exists to answer.

| Category | Condition |
|---|---|
| `DEVICE_ARTIFACT` | high full-cohort separation, collapses within the shared-device group |
| `RECOVERABLE_VIA_NORMALISATION` | flagged in §4 |
| `UNSTABLE_WITHIN_PERSON` | survives device control, but within-participant across-session spread rivals between-participant spread |
| `UNTESTABLE_NO_SHARED_DEVICE` | no device model with >=2 participants covers this feature's top separating group |
| `GENUINE_CANDIDATE` | survives both controls |
| `INSUFFICIENT_DATA` | too few observations in relevant groups to compute anything |

In [ ]:
def auc_two_groups(a, b, minn=10):
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    if len(a) < minn or len(b) < minn:
        return np.nan
    combined = np.r_[a, b]
    if np.std(combined) == 0:
        return np.nan
    r = rankdata(combined); n1 = len(a)
    u = r[:n1].sum() - n1 * (n1 + 1) / 2
    return max(u / (n1 * len(b)), 1 - u / (n1 * len(b)))

def full_cohort_auc(col):
    """Pairwise AUC across all participant pairs with enough data, averaged — a rough
    'does this separate people at all, ignoring device' signal, used only to decide
    whether a feature is worth running through the controls below."""
    pids = df.participantId.unique()
    scores = []
    for i in range(len(pids)):
        for j in range(i + 1, len(pids)):
            a = df.loc[df.participantId == pids[i], col].dropna().values.astype(float)
            b = df.loc[df.participantId == pids[j], col].dropna().values.astype(float)
            s = auc_two_groups(a, b)
            if not np.isnan(s):
                scores.append(s)
    return np.mean(scores) if scores else np.nan

def within_participant_stability(col, min_sessions=2):
    """For participants with >=2 sessions: AUC between their own sessions. High values
    mean the feature swings a lot within one person across time — exactly the transient-
    state (pace/fatigue) failure mode, independent of any device question."""
    multi = cohort[cohort.sessions >= min_sessions].index.tolist()
    scores = []
    for pid in multi:
        sids = df.loc[df.participantId == pid, "sessionId"].unique()
        if len(sids) != 2:
            continue
        a = df.loc[(df.participantId == pid) & (df.sessionId == sids[0]), col].dropna().values.astype(float)
        b = df.loc[(df.participantId == pid) & (df.sessionId == sids[1]), col].dropna().values.astype(float)
        s = auc_two_groups(a, b)
        if not np.isnan(s):
            scores.append(s)
    return np.mean(scores) if scores else np.nan

def classify_feature(col, recoverable_cols):
    device_auc = None
    if TESTABLE_MODELS:
        model = TESTABLE_MODELS[0]  # only one currently exists; loop-ready for when more do
        sub = df[df.deviceModel == model]
        pids = sub.participantId.unique()
        if len(pids) == 2:
            a = sub.loc[sub.participantId == pids[0], col].dropna().values.astype(float)
            b = sub.loc[sub.participantId == pids[1], col].dropna().values.astype(float)
            device_auc = auc_two_groups(a, b)

    within_auc = within_participant_stability(col)
    cohort_auc = full_cohort_auc(col)

    if device_auc is None or np.isnan(device_auc):
        status = "UNTESTABLE_NO_SHARED_DEVICE"
    elif device_auc < 0.60:
        status = "DEVICE_ARTIFACT" if col not in recoverable_cols else "RECOVERABLE_VIA_NORMALISATION"
    elif not np.isnan(within_auc) and within_auc >= 0.75:
        status = "UNSTABLE_WITHIN_PERSON"
    else:
        status = "GENUINE_CANDIDATE"

    return dict(col=col, cohort_auc=cohort_auc, device_auc=device_auc,
                within_person_auc=within_auc, status=status)

recoverable_cols = [r["col"] for r in recoverable]
decomposition = pd.DataFrame([classify_feature(c, recoverable_cols) for c in LIVE_COLS])
print(decomposition.status.value_counts().to_string())

status
GENUINE_CANDIDATE              252
DEVICE_ARTIFACT                171
UNTESTABLE_NO_SHARED_DEVICE     13
UNSTABLE_WITHIN_PERSON           8


In [ ]:
print("=== GENUINE_CANDIDATE features, ranked by cohort separation ===")
print(decomposition[decomposition.status == "GENUINE_CANDIDATE"]
      .sort_values("cohort_auc", ascending=False).head(20)
      .to_string(index=False, float_format=lambda x: f"{x:.3f}"))

=== GENUINE_CANDIDATE features, ranked by cohort separation ===
                             col  cohort_auc  device_auc  within_person_auc            status
     approval_swipe_duration_max       0.938       0.812                NaN GENUINE_CANDIDATE
              pointer_height_p95       0.920       0.883              0.535 GENUINE_CANDIDATE
               pointer_width_p95       0.920       0.883              0.535 GENUINE_CANDIDATE
            pointer_width_median       0.916       0.819              0.527 GENUINE_CANDIDATE
           pointer_height_median       0.916       0.819              0.527 GENUINE_CANDIDATE
              pointer_height_max       0.914       0.871              0.534 GENUINE_CANDIDATE
               pointer_width_max       0.914       0.871              0.534 GENUINE_CANDIDATE
    approval_swipe_duration_mean       0.905       0.812                NaN GENUINE_CANDIDATE
              pointer_width_mean       0.896       0.894              0.541 GENUINE_CANDID

**Read this as:** only `GENUINE_CANDIDATE` and `RECOVERABLE_VIA_NORMALISATION` (after
transforming) are currently defensible inputs to a scorer. `UNTESTABLE_NO_SHARED_DEVICE`
will likely be the largest bucket by far — that is the honest state of a 6-participant,
mostly-one-device-each cohort, not a bug in the method. Re-run this section after any new
participant joins an existing device model; that is the only way this bucket shrinks.

One caveat that applies to every `GENUINE_CANDIDATE` and `DEVICE_ARTIFACT` label right
now: with only one shared-device pair available, `device_auc` is a single comparison, not
an average over several. A feature passing that one test has cleared the bar this cohort
can set, not a bar proven to generalise — treat this section's labels as the current best
evidence, due for revision the moment a second shared-device pair exists.

## 6 — Within-family redundancy

**Goal:** for every base signal that produces the full multi-statistic suite (mean,
median, std, iqr, p95, slope, slope_r2, burstiness, local_inconsistency, ...), measure how
independent those columns actually are — run across every family, not spot-checked on
one or two.

In [ ]:
import re

def base_signal(col):
    for suffix in ["_mean","_std","_median","_iqr","_p95","_max","_min","_n","_cv",
                   "_burstiness","_local_inconsistency","_early_late_diff","_slope","_slope_r2"]:
        if col.endswith(suffix):
            return col[: -len(suffix)]
    return None

families = {}
for c in LIVE_COLS:
    b = base_signal(c)
    if b:
        families.setdefault(b, []).append(c)
families = {b: cols for b, cols in families.items() if len(cols) >= 4}

rows = []
for base, cols in families.items():
    sub = df[cols].apply(pd.to_numeric, errors="coerce")
    sub = sub.loc[:, sub.notna().mean() > 0.3]
    if sub.shape[1] < 4 or sub.dropna().shape[0] < 20:
        continue
    corr = sub.dropna().corr().abs()
    iu = np.triu_indices_from(corr.values, k=1)
    mean_abs_r = float(np.nanmean(corr.values[iu]))
    rows.append(dict(base_signal=base, n_stats=sub.shape[1], mean_abs_corr=round(mean_abs_r, 3)))

redundancy = pd.DataFrame(rows).sort_values("mean_abs_corr", ascending=False)
print(f"{len(redundancy)} base signals with >=4 usable statistics")
print(redundancy.to_string(index=False))

29 base signals with >=4 usable statistics
              base_signal  n_stats  mean_abs_corr
         pointer_pressure        7          0.533
            pointer_width       12          0.513
           pointer_height       12          0.513
 typing_press_to_press_ms        7          0.478
      typing_delta_length       12          0.472
          typing_dwell_ms       12          0.452
      typing_value_length       12          0.445
          pointer_hold_ms       13          0.441
            touch_radiusX       12          0.415
            touch_hold_ms       13          0.379
               motion_agy       13          0.370
motion_rotation_magnitude       13          0.363
         motion_magnitude       13          0.358
              touch_speed       13          0.353
               motion_agz       13          0.335
                motion_ay       11          0.326
              touch_force       13          0.317
                motion_ax       11          0.315
    ori

**Read this as:** a high `mean_abs_corr` means most of that base signal's 14
statistics move together — a handful would capture nearly the same information as all 14.
A low value means the shape statistics (burstiness, slope_r2, local_inconsistency) are
carrying genuinely separate information and should not be casually dropped for that
family, even if they are for another.

## 7 — Context questionnaire: variance and association

**Goal:** which `ctx*` fields vary at all in the current cohort, and — for the ones that
do — whether they associate with anything measured. Degrades gracefully to a no-op if
this file has zero `ctx*` columns (predates the questionnaire).

In [ ]:
if not CTX_COLS:
    print("No ctx* columns present in this file — skipping. Re-run once a pipeline build "
          "with the context questionnaire extraction has been synced.")
else:
    variance_report = []
    for c in CTX_COLS:
        s = df[c]
        variance_report.append(dict(col=c, n_unique=s.nunique(dropna=True),
                                     values=sorted(s.dropna().unique().tolist())[:6]))
    vr = pd.DataFrame(variance_report)
    print("=== ctx* field variance (computed live) ===")
    print(vr.to_string(index=False))

    varying = vr[vr.n_unique > 1].col.tolist()
    constant = vr[vr.n_unique <= 1].col.tolist()
    print(f"\nvarying: {varying}")
    print(f"constant so far (uninformative as a covariate): {constant}")

No ctx* columns present in this file — skipping. Re-run once a pipeline build with the context questionnaire extraction has been synced.

**Read this as:** a constant `ctx*` field isn't broken — it means this cohort hasn't
yet produced variation on that question (e.g. everyone sober, everyone seated). Only
fields in `varying` are candidates for the fusion-weighting layer right now.

## 8 — Modelling-readiness scorecard

**Goal:** tie every prior section into the arithmetic that determines what is trainable
right now, computed live so it updates automatically as more sessions land — rather than
hand-derived once and quoted from memory afterward.

In [ ]:
evaluable = cohort[cohort.sessions >= 2]
print(f"Evaluable identities (>=2 sessions, session-safe split): {len(evaluable)} of {len(cohort)}")
print(evaluable[["sessions", "windows"]].to_string())

Evaluable identities (>=2 sessions, session-safe split): 3 of 6
               sessions  windows
participantId                   
p3YDMHG               2      281
pA6XL23               2      249
pG5G4MS               2      105


In [ ]:
FAMILY_FLAGS = {"typing": "hasTyping", "touch": "hasTouch", "pointer": "hasPointer",
                 "scroll": "hasScroll", "motion": "hasMotion", "orientation": "hasOrientation"}

enrol_df = df[df.participantId.isin(evaluable.index)]
scorecard = []
for fam, flag in FAMILY_FLAGS.items():
    if flag not in df.columns:
        continue
    p = len([c for c in LIVE_COLS if c.startswith(fam + "_")])
    active_n = int((enrol_df[flag] == True).sum())
    ratio = active_n / max(p, 1)
    tier = ("full covariance" if ratio >= 3 else
            "shrinkage" if ratio >= 1.2 else
            "diagonal only")
    scorecard.append(dict(modality=fam, live_features=p, active_enrol_windows=active_n,
                           n_over_p=round(ratio, 2), estimator_tier=tier))

sc = pd.DataFrame(scorecard).sort_values("n_over_p")
print(sc.to_string(index=False))

   modality  live_features  active_enrol_windows  n_over_p  estimator_tier
     typing             62                   239      3.85 full covariance
     motion            150                   632      4.21 full covariance
    pointer             66                   584      8.85 full covariance
      touch             59                   585      9.92 full covariance
orientation             52                   632     12.15 full covariance
     scroll             17                   311     18.29 full covariance


**Read this as:** this table is the one to re-run after every pipeline rebuild. As
`evaluable` grows, `n_over_p` per modality grows with it, and estimator tiers should
migrate from `diagonal only` toward `full covariance` — track that migration directly
here rather than re-deriving it by hand each time.

## 9 — Known gaps this notebook cannot close

Stated explicitly, not left implicit:

- **Pace/tempo.** A participant's task-completion pace relative to design target requires
  `expectedSeconds` from `tasks.js` and the `_01`/`_02` task-round suffix, neither of
  which survives into `windows.csv`. This notebook cannot control for pace on its own —
  §5's `UNSTABLE_WITHIN_PERSON` category is the nearest available proxy, since a feature
  driven by session-to-session pace swings will show up there, but it cannot distinguish
  *why* a feature is unstable within a person, only *that* it is.
- **Task round.** `build_windows_dataset.py`'s `base_task_id()` strips the round suffix
  before it reaches this file. Any round-specific drift (round 2 running faster on some
  tasks) is invisible here.
- **Independent-sample count.** Windows overlap substantially (7.5s window / 2.5s stride).
  Every variance, AUC, or interval computed in this notebook is at the *window* level, not
  the independent-sample level — treat spreads as narrower than the true uncertainty,
  particularly in §5 and §6 where n is already small.
- **§5's device control is currently a single pair.** One shared-device group proves the
  method works; it does not yet prove any individual `GENUINE_CANDIDATE` feature
  generalises beyond that one pair. Re-running this notebook as more shared-device data
  arrives is the single highest-value way to strengthen §5's output — more valuable, for
  this specific question, than additional sessions from participants already in the
  cohort.